# Session 1 — LLM Foundations

Setup cell below.

In [ ]:
import os
from google import genai

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
api_key=os.environ.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=api_key
)

generation_config = {
    'temperature': 1,
    'max_output_tokens': 100,
    'top_p': 0.95,
    'thinking_level': 'low',
}

interaction = client.interactions.create(
    model='models/gemini-3-flash-preview',
    input='hello',
    generation_config=generation_config,
)


type='model_output' content=[TextContent(text='Hello! How can I help you today?', type='text', annotations=None)] error=None


In [4]:
print(interaction)

status='completed' model='models/gemini-3-flash-preview' agent=None id='v1_ChdsVnBJYW9DYUh1Zno0LUVQdHB6ZDBRMBIXbFZwSWFvQ2FIdWZ6NC1FUHRwemQwUTA' created='2026-07-04T00:57:57Z' updated='2026-07-04T00:57:57Z' system_instruction=None tools=None usage=Usage(total_input_tokens=2, input_tokens_by_modality=[ModalityTokens(modality='text', tokens=2)], total_cached_tokens=0, cached_tokens_by_modality=None, total_output_tokens=9, output_tokens_by_modality=None, total_tool_use_tokens=0, tool_use_tokens_by_modality=None, total_thought_tokens=0, total_tokens=11, grounding_tool_count=None) response_modalities=None response_mime_type=None previous_interaction_id=None environment_id=None service_tier='standard' webhook_config=None steps=[ThoughtStep(type='thought', signature='EjQKMgEMOdbHW3b+iCHtONdc0cJ3nZ36VEvlheoZfBeLTD3oO3DLQkzvf+2JJIMxse9WXtuV', summary=None), ModelOutputStep(type='model_output', content=[TextContent(text='Hello! How can I help you today?', type='text', annotations=None)], error=No

In [5]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

count_tokens("Hello")

1

## Byte Pair Encoding

Toy BPE: start from characters, repeatedly merge the most frequent adjacent pair until vocab budget used up. 
Watch characters fuse into chunks below.

In [9]:
from collections import Counter
def get_pairs(tokens):
    return Counter(zip(tokens, tokens[1:]))
    
def merge(tokens, pair):
    merged, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(tokens[i] + tokens[i + 1]); 
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return merged
    
tokens = list("low lower lowest")
for step in range(5):
    pairs = get_pairs(tokens)
    print(pairs)
    if not pairs:      
        break
    best = pairs.most_common(1)[0][0]
    tokens = merge(tokens, best)
    print(step, "->", tokens)

Counter({('l', 'o'): 3, ('o', 'w'): 3, (' ', 'l'): 2, ('w', 'e'): 2, ('w', ' '): 1, ('e', 'r'): 1, ('r', ' '): 1, ('e', 's'): 1, ('s', 't'): 1})
0 -> ['lo', 'w', ' ', 'lo', 'w', 'e', 'r', ' ', 'lo', 'w', 'e', 's', 't']
Counter({('lo', 'w'): 3, (' ', 'lo'): 2, ('w', 'e'): 2, ('w', ' '): 1, ('e', 'r'): 1, ('r', ' '): 1, ('e', 's'): 1, ('s', 't'): 1})
1 -> ['low', ' ', 'low', 'e', 'r', ' ', 'low', 'e', 's', 't']
Counter({(' ', 'low'): 2, ('low', 'e'): 2, ('low', ' '): 1, ('e', 'r'): 1, ('r', ' '): 1, ('e', 's'): 1, ('s', 't'): 1})
2 -> ['low', ' low', 'e', 'r', ' low', 'e', 's', 't']
Counter({(' low', 'e'): 2, ('low', ' low'): 1, ('e', 'r'): 1, ('r', ' low'): 1, ('e', 's'): 1, ('s', 't'): 1})
3 -> ['low', ' lowe', 'r', ' lowe', 's', 't']
Counter({('low', ' lowe'): 1, (' lowe', 'r'): 1, ('r', ' lowe'): 1, (' lowe', 's'): 1, ('s', 't'): 1})
4 -> ['low lowe', 'r', ' lowe', 's', 't']


# temperature, top_p, top_k

In [ ]:
from google.genai import types
MODEL = "gemini-2.5-flash-lite"

config = types.GenerateContentConfig(
    temperature=0.01,      # Avoids potential API fallback bugs with literal 0
    top_p=1.0,             # Includes full distribution but filtered by top_k
    top_k=1,               # Greedily forces the model to pick only the top 1 token
    candidate_count=1,     # Generates exactly one response sequence
    max_output_tokens=30,   # Prevents truncation variations
)
resp = client.models.generate_content(
        model=MODEL, contents="write a poem about llm",
        config=config)       
print( "->", resp.text)

# -> A whisper born of silicon and code,
# A vast expanse where knowledge is bestowed.
# Not flesh and blood, nor heart that beats with fire

## Prompt caching (cache_control)

Repeated prefix (system prompt, long doc, few-shot examples) costs tokens every call. Caching lets provider skip reprocessing that prefix on next call: cheaper + faster.

Two shapes for same idea:
- **Gemini**: explicit object. Create a `CachedContent` via `client.caches.create(...)`, get handle back, pass `cached_content=` on generate calls.
- **Anthropic**: inline flag. Mark a content block `cache_control: {"type": "ephemeral"}`, provider caches automatically, no separate object to manage.


In [ ]:
from google.genai import types

long_system_prompt = "You are a helpful assistant. " * 500  # needs min token count to be cache-eligible

cache = client.caches.create(
    model=MODEL,
    config=types.CreateCachedContentConfig(
        system_instruction=long_system_prompt,
        ttl="300s",
    ),
)

# Subsequent calls reference the cache handle instead of resending the prefix
cached_resp = client.models.generate_content(
    model=MODEL,
    contents="Summarize your role in one sentence.",
    config=types.GenerateContentConfig(cached_content=cache.name),
)
print(cached_resp.usage_metadata)


In [ ]:
import anthropic

anthropic_client = anthropic.Anthropic()

# cache_control marks a breakpoint: everything up to and including this block
# is cached for reuse on the next call within the TTL (default 5 min)
response = anthropic_client.messages.create(
    model="claude-sonnet-5",
    max_tokens=100,
    system=[
        {
            "type": "text",
            "text": long_system_prompt,
            "cache_control": {"type": "ephemeral"},
        }
    ],
    messages=[{"role": "user", "content": "Summarize your role in one sentence."}],
)
print(response.usage)  # cache_creation_input_tokens vs cache_read_input_tokens


## LiteLLM vs LangChain — why not just call the provider SDK?

Calling `client.models.generate_content(...)` directly is fine for a single-provider prototype. It becomes a
liability once you need to: swap providers, add a fallback model, or track cost/usage centrally — each provider
SDK has its own call shape, its own message format, its own error types.

- **LiteLLM**: thin unified call layer. One function (`completion`), swap the `model=` string to hit a different
  provider — same call shape, same response shape. Minimal magic on top.
- **LangChain**: same unification, but adds chains/agents/memory/tool-orchestration on top — heavier, worth it
  once you need those, overkill for a single call.


In [1]:
import litellm

# Same call shape regardless of provider — only the model string changes
resp = litellm.completion(
    model="gemini/gemini-2.5-flash-lite",
    messages=[{"role": "user", "content": "Say hi in 5 words."}],
)
print(resp.choices[0].message.content)

# resp = litellm.completion(model="claude-sonnet-5", messages=[...])   # same shape, different provider
# resp = litellm.completion(model="gpt-4o", messages=[...])            # same shape, different provider


Hello there, friend!


In [ ]:
from langchain.chat_models import init_chat_model

# init_chat_model gives the same .invoke() interface no matter the provider
gemini_chat = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")
claude_chat = init_chat_model("claude-sonnet-5", model_provider="anthropic")

print(gemini_chat.invoke("Say hi in 5 words.").content)
# print(claude_chat.invoke("Say hi in 5 words.").content)  # identical call shape


## Real-world best practices for LLM calls

Prototype code calls the model and hopes for the best. Production code has to survive rate limits, transient
failures, and models that occasionally don't follow instructions. Core reliability set:

1. **Retries + backoff** — transient 429/5xx need exponential backoff, not an immediate fail.
2. **Timeouts** — always set an explicit timeout; a hung request blocks a worker indefinitely.
3. **Error handling** — catch rate-limit / auth / content-filter errors distinctly, don't blanket `except Exception`.
4. **Rate-limit handling** — respect `Retry-After`, queue or shed load rather than hammering the API.
5. **Structured output validation** — validate the model's JSON/tool-call output against a schema before trusting
   it downstream; models occasionally violate schema even with tool-calling enabled.
6. **Logging/observability** — log prompt, response, token usage, and latency per call (redact sensitive data) so
   failures, cost, and regressions are debuggable after the fact.
